# 06c. RQ2 small-sample forecast shrinkage

This notebook is independent of Notebook 06 and 06b.

Purpose:
- keep the selected forecasting specification from Notebook 06 fixed;
- refit that fixed specification separately at rolling origins Years 4–7;
- apply forecast-level shrinkage using historical Kish effective sample size only;
- choose shrinkage strength (`lambda`) using rolling validation only;
- lock `lambda` before evaluating the untouched Year 8 test;
- report sensitivity for observed Year-8 / validation cells with `n_eff < 20`, `< 30`, and `< 50`.

In [33]:
import ast
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 2026
ROLLING_ORIGINS = [4, 5, 6, 7]
TEST_YEAR = 8
ALR_EPSILON = 1e-6

LAM_CANDIDATES = [0, 2, 5, 10, 20, 50, 100, 200]
PRIMARY_NEFF_THRESHOLD = 30
NEFF_THRESHOLDS = [20, 30, 50]

AGE_DATA_DIR = Path(r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Shuhan Zhao\q3")
DISABILITY_DATA_DIR = Path(r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\data\processed")
MODEL_OUTPUT_DIR = Path(r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\outputs")
SHRINKAGE_OUTPUT_DIR = MODEL_OUTPUT_DIR / "small_sample_shrinkage"
SHRINKAGE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary = pd.read_csv(MODEL_OUTPUT_DIR / "summary_full.csv")
best_params_df = pd.read_csv(MODEL_OUTPUT_DIR / "best_hyperparameters.csv")

print("Rolling origins:", ROLLING_ORIGINS)
print("Untouched test year:", TEST_YEAR)
print("Lambda candidates:", LAM_CANDIDATES)
print("Primary small-sample threshold:", PRIMARY_NEFF_THRESHOLD)
print("Outputs:", SHRINKAGE_OUTPUT_DIR)


Rolling origins: [4, 5, 6, 7]
Untouched test year: 8
Lambda candidates: [0, 2, 5, 10, 20, 50, 100, 200]
Primary small-sample threshold: 30
Outputs: C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\outputs\small_sample_shrinkage


## 1. Shared transformations and the exact Notebook 06 model specification


In [34]:
def shares_to_alr(shares):
    clipped = np.clip(np.asarray(shares, dtype=float), ALR_EPSILON, 1)
    clipped = clipped / clipped.sum(axis=1, keepdims=True)
    return np.log(clipped[:, :2] / clipped[:, [2]])


def alr_to_shares(values):
    values = np.clip(np.asarray(values, dtype=float), -30, 30)
    exponent = np.exp(values)
    denominator = 1 + exponent.sum(axis=1, keepdims=True)
    return np.column_stack([exponent / denominator, 1 / denominator])


def add_lag_features(frame, panel_keys, value_cols, lags=(1,), rolling_window=None, covid_years=(5, 6)):
    prepared = frame.sort_values(panel_keys + ["year"]).reset_index(drop=True).copy()
    grouped = prepared.groupby(panel_keys, sort=False)

    for column in value_cols:
        for lag in lags:
            prepared[f"{column}_lag{lag}"] = grouped[column].shift(lag)

        if rolling_window:
            prepared[f"{column}_roll{rolling_window}"] = grouped[column].transform(
                lambda s: s.shift(1).rolling(rolling_window, min_periods=1).mean()
            )

    prepared["time_trend"] = prepared["year"] / prepared["year"].max()
    prepared["is_covid_year"] = prepared["year"].isin(covid_years).astype(int)
    return prepared


def add_interaction_terms(frame, group_col, time_col="time_trend"):
    frame = frame.copy()
    dummies = pd.get_dummies(frame[group_col], prefix=f"{group_col}_x_time")
    interaction_cols = list(dummies.columns)
    frame[interaction_cols] = dummies.mul(frame[time_col], axis=0)
    return frame, interaction_cols


def get_eligible_split(prepared, target_cols, weight_col, lag_cols, before_year=None, at_year=None):
    if (before_year is None) == (at_year is None):
        raise ValueError("Provide exactly one of before_year or at_year.")

    if before_year is not None:
        rows = prepared[prepared["year"] < before_year]
        required = target_cols + [weight_col]
    else:
        rows = prepared[prepared["year"] == at_year]
        required = target_cols + lag_cols

    return rows.dropna(subset=required).copy()


def build_model(model_name, parameters, numeric_features, categorical_features, multi_output):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]

    if model_name == "Ridge Regression":
        numeric_steps.append(("scale", StandardScaler()))

    preprocess = ColumnTransformer([
        ("numeric", Pipeline(numeric_steps), numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ])

    if model_name == "Ridge Regression":
        estimator = Ridge(**parameters)
    elif model_name == "Random Forest":
        estimator = RandomForestRegressor(
            **parameters, random_state=RANDOM_STATE, n_jobs=-1
        )
    elif model_name == "Gradient Boosting":
        base = GradientBoostingRegressor(
            **parameters, random_state=RANDOM_STATE, loss="huber"
        )
        estimator = MultiOutputRegressor(base) if multi_output else base
    else:
        raise ValueError(f"Unsupported fitted model: {model_name}")

    return Pipeline([("preprocess", preprocess), ("model", estimator)])


def score_predictions(actual, predicted, is_composition):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    if is_composition:
        valid = np.isfinite(actual).all(axis=1) & np.isfinite(predicted).all(axis=1)
        actual = actual[valid]
        predicted = predicted[valid]
        if len(actual) == 0:
            return np.nan, 0
        tv = np.mean(0.5 * np.abs(actual - predicted).sum(axis=1))
        return float(tv), len(actual)

    actual = actual.ravel()
    predicted = predicted.ravel()
    valid = np.isfinite(actual) & np.isfinite(predicted)
    actual = actual[valid]
    predicted = predicted[valid]
    if len(actual) == 0:
        return np.nan, 0
    return float(mean_absolute_error(actual, predicted)), len(actual)


## 2. Read the fixed model family and fixed hyperparameters selected in Notebook 06



In [35]:
PARAMETER_COLUMNS = {
    "alpha",
    "n_estimators",
    "max_depth",
    "min_samples_leaf",
    "max_features",
    "learning_rate",
}


def selected_model_for_task(task_name, selection_metric):
    candidates = summary[
        (summary["task"] == task_name) &
        (summary["split"] == "validation")
    ].copy()

    if candidates.empty:
        raise ValueError(f"No Notebook 06 validation rows found for {task_name}")

    if selection_metric not in candidates.columns:
        raise ValueError(f"{selection_metric} missing from summary_full.csv for {task_name}")

    candidates = candidates.dropna(subset=[selection_metric])
    if candidates.empty:
        raise ValueError(f"No valid {selection_metric} values for {task_name}")

    return candidates.loc[candidates[selection_metric].idxmin(), "model"]


def fixed_params_for_task(task_name, model_name):
    if model_name == "Naive baseline":
        return {}

    short_key_map = {
        "age_overall_level": "age_overall",
        "disability_overall_level": "dis_overall",
        "disability_activity_level": "dis_level",
        "disability_months12": "months12",
        "disability_days10p60gr": "days",
        "age_months12": "age_months12",
        "age_days10p60gr": "age_days",
        "age_activity_level": "age_level",
    }
    short_key = short_key_map[task_name]

    row = best_params_df[
        (best_params_df["task"] == short_key) &
        (best_params_df["model"] == model_name)
    ]

    if len(row) != 1:
        raise ValueError(
            f"Expected exactly one best-parameter row for {task_name} / {model_name}; found {len(row)}"
        )

    row = row.iloc[0]
    params = {}

    for col in PARAMETER_COLUMNS:
        if col in row.index and pd.notna(row[col]):
            value = row[col]

            if col in {"n_estimators", "max_depth", "min_samples_leaf"}:
                value = int(value)
            else:
                value = float(value)

            params[col] = value

    return params


## 3. Historical support and forecast-level shrinkage


In [36]:
def add_historical_neff_support(frame, panel_keys, n_eff_col):
    result = frame.sort_values(panel_keys + ["year"]).reset_index(drop=True).copy()

    numeric_neff = pd.to_numeric(result[n_eff_col], errors="coerce")
    result["_numeric_neff"] = numeric_neff

    result["support_neff"] = (
        result.groupby(panel_keys, sort=False)["_numeric_neff"]
        .transform(lambda s: s.shift(1).expanding(min_periods=1).median())
    )

    result["observed_neff"] = result["_numeric_neff"]
    result = result.drop(columns=["_numeric_neff"])
    return result


def add_leave_one_borough_out_reference(pred_df, group_keys, pred_cols):
    result = pred_df.copy()

    grouped = result.groupby(group_keys, dropna=False)

    for col in pred_cols:
        group_sum = grouped[col].transform("sum")
        group_count = grouped[col].transform("count")

        loo = (group_sum - result[col]) / (group_count - 1)
        loo = loo.where(group_count > 1)

        result[f"london_ref_{col}"] = loo

    return result


def apply_forecast_shrinkage(pred_df, group_keys, pred_cols, lam):
    result = add_leave_one_borough_out_reference(pred_df, group_keys, pred_cols)

    support = pd.to_numeric(result["support_neff"], errors="coerce").to_numpy(dtype=float)
    valid_support = np.isfinite(support) & (support >= 0)

    if lam == 0:
        alpha = np.ones(len(result), dtype=float)
    else:
        alpha = np.ones(len(result), dtype=float)
        alpha[valid_support] = support[valid_support] / (support[valid_support] + lam)

    result["shrinkage_alpha"] = alpha

    for col in pred_cols:
        ref = pd.to_numeric(result[f"london_ref_{col}"], errors="coerce").to_numpy(dtype=float)
        own = pd.to_numeric(result[col], errors="coerce").to_numpy(dtype=float)

        can_shrink = valid_support & np.isfinite(ref)

        adjusted = own.copy()
        adjusted[can_shrink] = (
            alpha[can_shrink] * own[can_shrink] +
            (1 - alpha[can_shrink]) * ref[can_shrink]
        )
        result[f"adjusted_{col}"] = adjusted

    if len(pred_cols) == 3:
        adjusted_cols = [f"adjusted_{c}" for c in pred_cols]
        arr = result[adjusted_cols].to_numpy(dtype=float)
        row_sum = arr.sum(axis=1, keepdims=True)
        valid = np.isfinite(arr).all(axis=1) & (row_sum[:, 0] > 0)
        arr[valid] = arr[valid] / row_sum[valid]
        result[adjusted_cols] = arr

    return result


## 4. Task definitions


In [37]:
age_overall = pd.read_csv(AGE_DATA_DIR / "q3_age_overall_activity_level_panel.csv")
age_activity = pd.read_csv(AGE_DATA_DIR / "q3_age_activity_participation_level_panel_complete.csv")
disability_overall = pd.read_csv(DISABILITY_DATA_DIR / "RQ3_borough_disability_overall_all_years.csv")
disability_activity = pd.read_csv(DISABILITY_DATA_DIR / "RQ3_borough_disability_MEMS7GR_all_years.csv")
disability_rates = pd.read_csv(DISABILITY_DATA_DIR / "RQ3_borough_disability_days_months_all_years.csv")

for frame in [age_overall, age_activity, disability_overall, disability_activity, disability_rates]:
    frame["LA_2023"] = frame["LA_2023"].astype("Int64").astype(str)


def first_existing_column(frame, candidates):
    for candidate in candidates:
        if candidate in frame.columns:
            return candidate
    return None


TASKS = {
    "age_overall_level": {
        "frame": age_overall,
        "panel_keys": ["LA_2023", "age_group"],
        "target_cols": ["overall_inactive_rate", "overall_fairly_active_rate", "overall_active_rate"],
        "weight_col": "weighted_n_overall_activity_level",
        "n_eff_candidates": ["n_eff_overall_activity_level", "n_eff"],
        "feature_level": "overall",
        "is_composition": True,
        "selection_metric": "total_variation",
    },
    "disability_overall_level": {
        "frame": disability_overall,
        "panel_keys": ["LA_2023", "disability_group"],
        "target_cols": ["inactive_rate", "fairly_active_rate", "active_rate"],
        "weight_col": "weighted_n",
        "n_eff_candidates": ["n_eff"],
        "feature_level": "overall",
        "is_composition": True,
        "selection_metric": "total_variation",
    },
    "age_activity_level": {
        "frame": age_activity,
        "panel_keys": ["LA_2023", "age_group", "activity_suffix"],
        "target_cols": ["activity_inactive_rate", "activity_fairly_active_rate", "activity_active_rate"],
        "weight_col": "weighted_n_activity_level",
        "n_eff_candidates": ["n_eff_activity_level", "n_eff"],
        "feature_level": "activity",
        "is_composition": True,
        "selection_metric": "total_variation",
    },
    "disability_activity_level": {
        "frame": disability_activity,
        "panel_keys": ["LA_2023", "disability_group", "activity"],
        "target_cols": ["inactive_rate", "fairly_active_rate", "active_rate"],
        "weight_col": "weighted_n",
        "n_eff_candidates": ["n_eff"],
        "feature_level": "activity",
        "is_composition": True,
        "selection_metric": "total_variation",
    },
    "age_months12": {
        "frame": age_activity,
        "panel_keys": ["LA_2023", "age_group", "activity_suffix"],
        "target_cols": ["months12_rate"],
        "weight_col": "weighted_n_months12",
        "n_eff_candidates": ["n_eff_months12", "n_eff_MONTHS_12"],
        "feature_level": "activity",
        "is_composition": False,
        "selection_metric": "months12_rate_mae",
    },
    "disability_months12": {
        "frame": disability_rates,
        "panel_keys": ["LA_2023", "disability_group", "activity"],
        "target_cols": ["participation_MONTHS_12"],
        "weight_col": "weighted_n_MONTHS_12",
        "n_eff_candidates": ["n_eff_MONTHS_12", "n_eff_months12"],
        "feature_level": "activity",
        "is_composition": False,
        "selection_metric": "participation_MONTHS_12_mae",
    },
    "age_days10p60gr": {
        "frame": age_activity,
        "panel_keys": ["LA_2023", "age_group", "activity_suffix"],
        "target_cols": ["days10p60gr_rate"],
        "weight_col": "weighted_n_days10p60gr",
        "n_eff_candidates": ["n_eff_days10p60gr", "n_eff_DAYS10P60GR"],
        "feature_level": "activity",
        "is_composition": False,
        "selection_metric": "days10p60gr_rate_mae",
    },
    "disability_days10p60gr": {
        "frame": disability_rates,
        "panel_keys": ["LA_2023", "disability_group", "activity"],
        "target_cols": ["participation_DAYS10P60GR"],
        "weight_col": "weighted_n_DAYS10P60GR",
        "n_eff_candidates": ["n_eff_DAYS10P60GR", "n_eff_days10p60gr"],
        "feature_level": "activity",
        "is_composition": False,
        "selection_metric": "participation_DAYS10P60GR_mae",
    },
}

usable_tasks = {}
task_audit = []

for task_name, config in TASKS.items():
    n_eff_col = first_existing_column(config["frame"], config["n_eff_candidates"])
    selected_model = selected_model_for_task(task_name, config["selection_metric"])
    params = fixed_params_for_task(task_name, selected_model)

    status = "ready" if n_eff_col is not None else "skipped_missing_n_eff"
    task_audit.append({
        "task": task_name,
        "selected_model_from_06": selected_model,
        "n_eff_col": n_eff_col,
        "status": status,
    })

    if n_eff_col is not None:
        config = dict(config)
        config["n_eff_col"] = n_eff_col
        config["selected_model"] = selected_model
        config["fixed_params"] = params
        usable_tasks[task_name] = config

task_audit = pd.DataFrame(task_audit)
display(task_audit)

if not usable_tasks:
    raise RuntimeError("No task contains a usable Kish n_eff column.")


,task,selected_model_from_06,n_eff_col,status
0,age_overall_level,Gradient Boosting,n_eff_overall_activity_level,ready
1,disability_overall_level,Gradient Boosting,n_eff,ready
2,age_activity_level,Random Forest,n_eff_activity_level,ready
3,disability_activity_level,Gradient Boosting,n_eff,ready
4,age_months12,Naive baseline,n_eff_months12,ready
5,disability_months12,Ridge Regression,n_eff_MONTHS_12,ready
6,age_days10p60gr,Naive baseline,n_eff_days10p60gr,ready
7,disability_days10p60gr,Gradient Boosting,n_eff_DAYS10P60GR,ready


## 5. Produce leakage-free predictions at one historical origin



In [38]:
def prepare_task(config):
    frame = add_historical_neff_support(
        config["frame"],
        config["panel_keys"],
        config["n_eff_col"],
    )

    if config["feature_level"] == "overall":
        prepared = add_lag_features(
            frame, config["panel_keys"], config["target_cols"], lags=(1, 2)
        )
        extra_cols = [f"{c}_lag2" for c in config["target_cols"]]
    else:
        prepared = add_lag_features(
            frame, config["panel_keys"], config["target_cols"],
            lags=(1,), rolling_window=2
        )
        extra_cols = [f"{c}_roll2" for c in config["target_cols"]]

    prepared, interaction_cols = add_interaction_terms(
        prepared, config["panel_keys"][1]
    )

    lag1_cols = [f"{c}_lag1" for c in config["target_cols"]]
    lag_cols = lag1_cols + extra_cols
    base_numeric = lag_cols + ["time_trend", "is_covid_year"]

    return prepared, lag1_cols, lag_cols, base_numeric, interaction_cols


def prediction_frame_at_origin(task_name, config, origin):
    prepared, lag1_cols, lag_cols, base_numeric, interaction_cols = prepare_task(config)

    val = get_eligible_split(
        prepared,
        config["target_cols"],
        config["weight_col"],
        lag_cols,
        at_year=origin,
    )

    if val.empty:
        raise RuntimeError(f"{task_name}: no eligible rows at origin {origin}")

    model_name = config["selected_model"]

    if model_name == "Naive baseline":
        prediction = val[lag1_cols].to_numpy(dtype=float)

    else:
        train = get_eligible_split(
            prepared,
            config["target_cols"],
            config["weight_col"],
            lag_cols,
            before_year=origin,
        )

        if train.empty:
            raise RuntimeError(f"{task_name}: no training rows before origin {origin}")

        numeric = list(base_numeric)
        if model_name == "Ridge Regression":
            numeric += interaction_cols

        categorical = list(config["panel_keys"])
        feature_cols = numeric + categorical

        model = build_model(
            model_name,
            config["fixed_params"],
            numeric,
            categorical,
            multi_output=config["is_composition"],
        )

        train_weights = pd.to_numeric(
            train[config["weight_col"]], errors="coerce"
        ).to_numpy(dtype=float)

        if config["is_composition"]:
            model.fit(
                train[feature_cols],
                shares_to_alr(train[config["target_cols"]]),
                model__sample_weight=train_weights,
            )
            prediction = alr_to_shares(model.predict(val[feature_cols]))
        else:
            target = config["target_cols"][0]
            model.fit(
                train[feature_cols],
                train[target],
                model__sample_weight=train_weights,
            )
            prediction = np.clip(model.predict(val[feature_cols]), 0, 1).reshape(-1, 1)

    if prediction.ndim == 1:
        prediction = prediction.reshape(-1, 1)

    keep = (
        config["panel_keys"] +
        ["year", "support_neff", "observed_neff"]
    )

    out = val[keep].copy()

    for j, col in enumerate(config["target_cols"]):
        out[col] = prediction[:, j]
        out[f"{col}_actual"] = val[col].to_numpy(dtype=float)

    return out


## 6. Rolling validation: evaluate every lambda on Years 4–7 only


In [39]:
def evaluate_prediction_frame(pred_df, config, lam):
    group_keys = ["year"] + [
        key for key in config["panel_keys"] if key != "LA_2023"
    ]
    pred_cols = config["target_cols"]
    actual_cols = [f"{c}_actual" for c in pred_cols]

    adjusted = apply_forecast_shrinkage(
        pred_df,
        group_keys=group_keys,
        pred_cols=pred_cols,
        lam=lam,
    )

    adjusted_cols = [f"adjusted_{c}" for c in pred_cols]

    rows = []

    def append_score(subset_name, frame):
        if frame.empty:
            return
        score, n_obs = score_predictions(
            frame[actual_cols].to_numpy(dtype=float),
            frame[adjusted_cols].to_numpy(dtype=float),
            config["is_composition"],
        )
        rows.append({
            "lam": lam,
            "subset": subset_name,
            "error": score,
            "observations": n_obs,
        })

    append_score("all", adjusted)

    for threshold in NEFF_THRESHOLDS:
        mask = pd.to_numeric(
            adjusted["observed_neff"], errors="coerce"
        ) < threshold
        append_score(f"n_eff<{threshold}", adjusted[mask])

    return pd.DataFrame(rows), adjusted


validation_rows = []
validation_prediction_audit = []

for task_name, config in usable_tasks.items():
    print(f"Validation: {task_name} | fixed model = {config['selected_model']}")

    completed_origins = []

    for origin in ROLLING_ORIGINS:
        pred_df = prediction_frame_at_origin(task_name, config, origin)
        completed_origins.append(origin)

        for lam in LAM_CANDIDATES:
            scored, adjusted = evaluate_prediction_frame(pred_df, config, lam)
            scored["task"] = task_name
            scored["origin"] = origin
            scored["fixed_model"] = config["selected_model"]
            validation_rows.append(scored)

        validation_prediction_audit.append({
            "task": task_name,
            "origin": origin,
            "rows": len(pred_df),
            "median_historical_support_neff": float(
                pd.to_numeric(pred_df["support_neff"], errors="coerce").median()
            ),
            "median_observed_neff_for_evaluation_only": float(
                pd.to_numeric(pred_df["observed_neff"], errors="coerce").median()
            ),
        })

    if completed_origins != ROLLING_ORIGINS:
        raise RuntimeError(
            f"{task_name}: expected rolling origins {ROLLING_ORIGINS}, got {completed_origins}"
        )

validation_results = pd.concat(validation_rows, ignore_index=True)
validation_prediction_audit = pd.DataFrame(validation_prediction_audit)

display(validation_results.head())
display(validation_prediction_audit)


Validation: age_overall_level | fixed model = Gradient Boosting
Validation: disability_overall_level | fixed model = Gradient Boosting
Validation: age_activity_level | fixed model = Random Forest
Validation: disability_activity_level | fixed model = Gradient Boosting
Validation: age_months12 | fixed model = Naive baseline
Validation: disability_months12 | fixed model = Ridge Regression
Validation: age_days10p60gr | fixed model = Naive baseline
Validation: disability_days10p60gr | fixed model = Gradient Boosting


,lam,subset,error,observations,task,origin,fixed_model
0,0,all,0.117437,256,age_overall_level,4,Gradient Boosting
1,0,n_eff<20,0.222616,58,age_overall_level,4,Gradient Boosting
2,0,n_eff<30,0.176672,99,age_overall_level,4,Gradient Boosting
3,0,n_eff<50,0.136841,174,age_overall_level,4,Gradient Boosting
4,2,all,0.117053,256,age_overall_level,4,Gradient Boosting


,task,origin,rows,median_historical_support_neff,median_observed_neff_for_evaluation_only
0,age_overall_level,4,256,44.469070,39.036457
1,age_overall_level,5,256,42.981747,40.272299
2,age_overall_level,6,256,41.398599,41.907502
3,age_overall_level,7,256,41.224759,43.123529
4,disability_overall_level,4,491,10.708797,10.222997
5,disability_overall_level,5,492,10.466786,9.741792
6,disability_overall_level,6,496,10.222671,10.644178
7,disability_overall_level,7,497,10.265974,12.532750
8,age_activity_level,4,31744,44.469070,39.036457
9,age_activity_level,5,31744,42.981747,40.272299


In [40]:
validation_mean = (
    validation_results
    .groupby(["task", "fixed_model", "lam", "subset"], as_index=False)
    .agg(
        mean_origin_error=("error", "mean"),
        n_origins=("origin", "nunique"),
        total_scored_observations=("observations", "sum"),
    )
)

primary_subset = f"n_eff<{PRIMARY_NEFF_THRESHOLD}"

selection_rows = []

for task_name, task_frame in validation_mean.groupby("task"):
    primary = task_frame[task_frame["subset"] == primary_subset].copy()

    if primary.empty:
        raise RuntimeError(
            f"{task_name}: no validation cells available for primary subset {primary_subset}"
        )

    if (primary["n_origins"] != len(ROLLING_ORIGINS)).any():
        raise RuntimeError(
            f"{task_name}: {primary_subset} was not represented in all four rolling origins."
        )

    all_cells = (
        task_frame[task_frame["subset"] == "all"]
        [["lam", "mean_origin_error"]]
        .rename(columns={"mean_origin_error": "all_cell_mean_origin_error"})
    )

    primary = primary.merge(all_cells, on="lam", how="left")

    primary = primary.sort_values(
        ["mean_origin_error", "all_cell_mean_origin_error", "lam"],
        ascending=[True, True, True],
    )

    winner = primary.iloc[0]

    selection_rows.append({
        "task": task_name,
        "fixed_model": winner["fixed_model"],
        "selected_lambda": int(winner["lam"]),
        "primary_subset": primary_subset,
        "primary_validation_error": float(winner["mean_origin_error"]),
        "all_cell_validation_error": float(winner["all_cell_mean_origin_error"]),
        "n_validation_origins": int(winner["n_origins"]),
    })

lambda_selection = pd.DataFrame(selection_rows)

print("LAMBDA IS NOW LOCKED FROM YEARS 4-7 VALIDATION ONLY")
display(lambda_selection)

display(
    validation_mean.sort_values(["task", "subset", "lam"])
)


LAMBDA IS NOW LOCKED FROM YEARS 4-7 VALIDATION ONLY


,task,fixed_model,selected_lambda,primary_subset,primary_validation_error,all_cell_validation_error,n_validation_origins
0,age_activity_level,Random Forest,50,n_eff<30,0.006158,0.007176,4
1,age_days10p60gr,Naive baseline,100,n_eff<30,0.007826,0.007493,4
2,age_months12,Naive baseline,100,n_eff<30,0.014414,0.014469,4
3,age_overall_level,Gradient Boosting,20,n_eff<30,0.180407,0.113049,4
4,disability_activity_level,Gradient Boosting,20,n_eff<30,0.007983,0.007973,4
5,disability_days10p60gr,Gradient Boosting,20,n_eff<30,0.009317,0.008863,4
6,disability_months12,Ridge Regression,2,n_eff<30,0.022223,0.020748,4
7,disability_overall_level,Gradient Boosting,5,n_eff<30,0.232173,0.200054,4


,task,fixed_model,lam,subset,mean_origin_error,n_origins,total_scored_observations
0,age_activity_level,Random Forest,0,all,0.007314,4,126870
4,age_activity_level,Random Forest,2,all,0.007287,4,126870
8,age_activity_level,Random Forest,5,all,0.007256,4,126870
12,age_activity_level,Random Forest,10,all,0.007222,4,126870
16,age_activity_level,Random Forest,20,all,0.007187,4,126870
...,...,...,...,...,...,...,...
239,disability_overall_level,Gradient Boosting,10,n_eff<50,0.217229,4,1768
243,disability_overall_level,Gradient Boosting,20,n_eff<50,0.217431,4,1768
247,disability_overall_level,Gradient Boosting,50,n_eff<50,0.217765,4,1768
251,disability_overall_level,Gradient Boosting,100,n_eff<50,0.217996,4,1768


## 7. Untouched Year 8 test



In [41]:
test_rows = []
test_prediction_rows = []

for task_name, config in usable_tasks.items():
    selected_lambda = int(
        lambda_selection.loc[
            lambda_selection["task"] == task_name,
            "selected_lambda"
        ].iloc[0]
    )

    pred_df = prediction_frame_at_origin(task_name, config, TEST_YEAR)

    lambdas_to_test = sorted(set([0, selected_lambda]))

    for lam in lambdas_to_test:
        scored, adjusted = evaluate_prediction_frame(pred_df, config, lam)
        scored["task"] = task_name
        scored["year"] = TEST_YEAR
        scored["fixed_model"] = config["selected_model"]
        scored["validation_selected_lambda"] = selected_lambda
        scored["is_selected_lambda"] = (lam == selected_lambda)
        test_rows.append(scored)

        if lam == selected_lambda:
            adjusted["task"] = task_name
            adjusted["selected_lambda"] = selected_lambda
            test_prediction_rows.append(adjusted)

year8_results = pd.concat(test_rows, ignore_index=True)
year8_selected_predictions = pd.concat(test_prediction_rows, ignore_index=True)

display(
    year8_results.sort_values(["task", "subset", "lam"])
)


,lam,subset,error,observations,task,year,fixed_model,validation_selected_lambda,is_selected_lambda
16,0,all,0.008036,31585,age_activity_level,8,Random Forest,50,False
20,50,all,0.007898,31585,age_activity_level,8,Random Forest,50,True
17,0,n_eff<20,0.007115,7029,age_activity_level,8,Random Forest,50,False
21,50,n_eff<20,0.007018,7029,age_activity_level,8,Random Forest,50,True
18,0,n_eff<30,0.007574,11084,age_activity_level,8,Random Forest,50,False
...,...,...,...,...,...,...,...,...,...
13,5,n_eff<20,0.237659,315,disability_overall_level,8,Gradient Boosting,5,True
10,0,n_eff<30,0.219031,383,disability_overall_level,8,Gradient Boosting,5,False
14,5,n_eff<30,0.216530,383,disability_overall_level,8,Gradient Boosting,5,True
11,0,n_eff<50,0.203531,437,disability_overall_level,8,Gradient Boosting,5,False


## 8. Compact improvement table and save outputs



In [42]:
comparison_rows = []

for task_name, task_frame in year8_results.groupby("task"):
    selected_lambda = int(
        lambda_selection.loc[
            lambda_selection["task"] == task_name,
            "selected_lambda"
        ].iloc[0]
    )

    for subset in ["all"] + [f"n_eff<{t}" for t in NEFF_THRESHOLDS]:
        sub = task_frame[task_frame["subset"] == subset]

        raw = sub[sub["lam"] == 0]
        selected = sub[sub["lam"] == selected_lambda]

        if raw.empty or selected.empty:
            continue

        raw_error = float(raw["error"].iloc[0])
        selected_error = float(selected["error"].iloc[0])

        comparison_rows.append({
            "task": task_name,
            "fixed_model": sub["fixed_model"].iloc[0],
            "selected_lambda": selected_lambda,
            "subset": subset,
            "raw_year8_error": raw_error,
            "shrunken_year8_error": selected_error,
            "absolute_error_change": selected_error - raw_error,
            "relative_error_reduction_pct": (
                100 * (raw_error - selected_error) / raw_error
                if np.isfinite(raw_error) and raw_error != 0
                else np.nan
            ),
        })

year8_comparison = pd.DataFrame(comparison_rows)

display(
    year8_comparison.sort_values(["task", "subset"])
)

validation_results.to_csv(
    SHRINKAGE_OUTPUT_DIR / "validation_all_lambdas_by_origin.csv",
    index=False,
)
validation_mean.to_csv(
    SHRINKAGE_OUTPUT_DIR / "validation_all_lambdas_mean.csv",
    index=False,
)
lambda_selection.to_csv(
    SHRINKAGE_OUTPUT_DIR / "selected_lambda_by_task.csv",
    index=False,
)
year8_results.to_csv(
    SHRINKAGE_OUTPUT_DIR / "year8_raw_vs_selected_lambda_scores.csv",
    index=False,
)
year8_comparison.to_csv(
    SHRINKAGE_OUTPUT_DIR / "year8_shrinkage_improvement_summary.csv",
    index=False,
)
year8_selected_predictions.to_csv(
    SHRINKAGE_OUTPUT_DIR / "year8_selected_lambda_predictions_audit.csv",
    index=False,
)
task_audit.to_csv(
    SHRINKAGE_OUTPUT_DIR / "task_neff_availability_audit.csv",
    index=False,
)
validation_prediction_audit.to_csv(
    SHRINKAGE_OUTPUT_DIR / "validation_prediction_audit.csv",
    index=False,
)

print("Saved:")
for p in sorted(SHRINKAGE_OUTPUT_DIR.glob("*.csv")):
    print("-", p.name)


,task,fixed_model,selected_lambda,subset,raw_year8_error,shrunken_year8_error,absolute_error_change,relative_error_reduction_pct
0,age_activity_level,Random Forest,50,all,0.008036,0.007898,-0.000138,1.715483
1,age_activity_level,Random Forest,50,n_eff<20,0.007115,0.007018,-0.000098,1.374573
2,age_activity_level,Random Forest,50,n_eff<30,0.007574,0.007473,-0.000100,1.325206
3,age_activity_level,Random Forest,50,n_eff<50,0.007800,0.007618,-0.000182,2.334552
4,age_days10p60gr,Naive baseline,100,all,0.009106,0.007919,-0.001187,13.036631
5,age_days10p60gr,Naive baseline,100,n_eff<20,0.010257,0.008703,-0.001554,15.152542
6,age_days10p60gr,Naive baseline,100,n_eff<30,0.010032,0.008666,-0.001366,13.620288
7,age_days10p60gr,Naive baseline,100,n_eff<50,0.009384,0.008135,-0.001249,13.307535
8,age_months12,Naive baseline,100,all,0.017008,0.014680,-0.002328,13.688004
9,age_months12,Naive baseline,100,n_eff<20,0.017398,0.015590,-0.001808,10.394082


Saved:
- selected_lambda_by_task.csv
- task_neff_availability_audit.csv
- validation_all_lambdas_by_origin.csv
- validation_all_lambdas_mean.csv
- validation_prediction_audit.csv
- year8_raw_vs_selected_lambda_scores.csv
- year8_selected_lambda_predictions_audit.csv
- year8_shrinkage_improvement_summary.csv


## Acceptance checks

Before using the results in the thesis, confirm:

1. Every usable task has exactly four rolling validation origins.
2. `lambda` was selected using validation Years 4–7 only.
3. Year 8 was evaluated only after lambda selection.
4. `support_neff` always comes from years before the predicted year.
5. Target-year `observed_neff` is used only for retrospective error stratification.
6. No target-year `weighted_n` is used to construct the London shrinkage reference.
7. The London reference excludes the focal borough itself.
8. Lambda 0 reproduces the unshrunken fixed-model prediction.
9. Shrinkage is claimed to mitigate small-sample prediction instability only if held-out error actually improves, especially for the `<30` subset without materially damaging overall performance.
10. Tasks skipped because their upstream panel lacks `n_eff` must not be silently described as having undergone shrinkage.
